# Titanic baseline: logistic regression

This notebook inspects the supplied files, derives the target and submission schema from them, evaluates a leakage-safe preprocessing/model pipeline with fixed stratified cross-validation, and writes a verified submission.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_SPLITS = 5

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
data_dir = project_root / 'data' / 'raw'
submission_dir = project_root / 'submissions'
submission_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample_submission = pd.read_csv(data_dir / 'gender_submission.csv')

## File inspection and schema discovery

In [2]:
def profile_frame(name, frame):
    return pd.DataFrame({
        'file': name,
        'dtype': frame.dtypes.astype(str),
        'missing': frame.isna().sum(),
        'missing_pct': (100 * frame.isna().mean()).round(2),
        'unique': frame.nunique(dropna=False),
    })

for name, frame in [('train.csv', train), ('test.csv', test), ('gender_submission.csv', sample_submission)]:
    print(f'{name}: shape={frame.shape}, columns={frame.columns.tolist()}')
    display(profile_frame(name, frame))

train_only_columns = [column for column in train.columns if column not in test.columns]
assert len(train_only_columns) == 1, f'Expected exactly one training-only target column, found {train_only_columns}'
target_column = train_only_columns[0]
submission_columns = sample_submission.columns.tolist()
id_columns = [column for column in submission_columns if column != target_column]
assert len(id_columns) == 1, f'Expected one submission ID column, found {id_columns}'
id_column = id_columns[0]

print(f'Discovered target: {target_column}')
print(f'Discovered submission schema: {submission_columns}')
display(train[target_column].value_counts().sort_index().rename('count').to_frame().assign(
    proportion=train[target_column].value_counts(normalize=True).sort_index().round(4)
))

train.csv: shape=(891, 12), columns=['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,file,dtype,missing,missing_pct,unique
PassengerId,train.csv,int64,0,0.00,891
Survived,train.csv,int64,0,0.00,2
Pclass,train.csv,int64,0,0.00,3
Name,train.csv,str,0,0.00,891
Sex,train.csv,str,0,0.00,2
Age,train.csv,float64,177,19.87,89
SibSp,train.csv,int64,0,0.00,7
Parch,train.csv,int64,0,0.00,7
Ticket,train.csv,str,0,0.00,681
Fare,train.csv,float64,0,0.00,248


test.csv: shape=(418, 11), columns=['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,file,dtype,missing,missing_pct,unique
PassengerId,test.csv,int64,0,0.00,418
Pclass,test.csv,int64,0,0.00,3
Name,test.csv,str,0,0.00,418
Sex,test.csv,str,0,0.00,2
Age,test.csv,float64,86,20.57,80
SibSp,test.csv,int64,0,0.00,7
Parch,test.csv,int64,0,0.00,8
Ticket,test.csv,str,0,0.00,363
Fare,test.csv,float64,1,0.24,170
Cabin,test.csv,str,327,78.23,77


gender_submission.csv: shape=(418, 2), columns=['PassengerId', 'Survived']


,file,dtype,missing,missing_pct,unique
PassengerId,gender_submission.csv,int64,0,0.0,418
Survived,gender_submission.csv,int64,0,0.0,2


Discovered target: Survived
Discovered submission schema: ['PassengerId', 'Survived']


,count,proportion
Survived,,
0,549,0.6162
1,342,0.3838


## Modeling decisions from the EDA

- Use `Age`, `SibSp`, `Parch`, and `Fare` as numeric measurements.
- Treat `Pclass`, `Sex`, and `Embarked` as categorical, even though `Pclass` is integer-coded.
- Exclude `PassengerId` because it is an identifier.
- Defer `Name`, `Ticket`, and `Cabin`: they are high-cardinality/unstructured fields, and `Cabin` is mostly missing. This avoids Titanic-specific feature engineering in the first baseline.
- Impute, scale, and encode inside the pipeline so every validation fold learns preprocessing only from its training partition.

In [3]:
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']
model_features = numeric_features + categorical_features

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=SEED)),
])

X = train[model_features]
y = train[target_column]
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_scores = cross_val_score(baseline_pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=1)
cv_results = pd.DataFrame({'fold': np.arange(1, N_SPLITS + 1), 'accuracy': fold_scores})
display(cv_results.style.format({'accuracy': '{:.4f}'}))
print(f'Mean accuracy: {fold_scores.mean():.4f}')
print(f'Standard deviation: {fold_scores.std():.4f}')

,fold,accuracy
0,1,0.7821
1,2,0.8034
2,3,0.7978
3,4,0.7809
4,5,0.8202


Mean accuracy: 0.7969
Standard deviation: 0.0146


## Fit on all training rows and create the submission

In [4]:
baseline_pipeline.fit(X, y)
test_predictions = baseline_pipeline.predict(test[model_features])
submission = pd.DataFrame({
    id_column: test[id_column],
    target_column: test_predictions,
})[submission_columns]
submission_path = submission_dir / 'submission_01_logistic.csv'
submission.to_csv(submission_path, index=False)

written_submission = pd.read_csv(submission_path)
assert written_submission.columns.tolist() == submission_columns
assert len(written_submission) == len(sample_submission) == len(test)
assert written_submission[id_column].equals(sample_submission[id_column])
assert written_submission[id_column].equals(test[id_column])
assert written_submission[target_column].notna().all()
assert set(written_submission[target_column].unique()).issubset(set(train[target_column].unique()))
assert written_submission[target_column].dtype == sample_submission[target_column].dtype

print(f'Wrote and verified: {submission_path}')
print(f'Shape: {written_submission.shape}')
print(f'Prediction values: {sorted(written_submission[target_column].unique().tolist())}')
display(written_submission.head())

Wrote and verified: /Users/jeremy/Projects/kaggle-titanic/submissions/submission_01_logistic.csv
Shape: (418, 2)
Prediction values: [0, 1]


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
